# Phase 4a - Final detector training (Colab T4)

Trains both detectors at the configuration locked by Phases 2-3, for the
full **100 epochs** - the same budget as the Phase 1 baselines.

| | imgsz | lr0 | box / cls / dfl | decided by |
|---|---|---|---|---|
| Child | **416** | 0.002 | defaults | 3a re-run; auto rate for nc=1 |
| Hazard | **640** | 8.8e-4 | 8.14 / 0.75 / 1.06 | 3a + Phase 2 sweep |

Both use **AdamW** (Phase 3b). The config is baked into the script, not
passed here, so this run cannot silently disagree with those decisions.

## Why this run matters beyond producing weights

It is the **first fair test of whether the Phase 2 tuning helped at all.**
The sweep used 20-epoch schedules, and ultralytics anneals the learning rate
across the *scheduled* epoch count - so a 20-epoch run is fully annealed at
epoch 20 while a 100-epoch run is only a fifth through its decay. Comparing
them at epoch 20 favours the short run on schedule alone. Only a 100-epoch
tuned run can be compared against the 100-epoch baseline. The script prints
the delta at the end.

**Budget:** child ~0.9 h, hazard ~2.4 h (**~3.3 h total**). Splitting across
two sessions is safer than one long one. Resumable either way.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics==8.4.106 roboflow

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
FINAL_PROJECT = "/content/drive/MyDrive/deeplrn_group2/runs"
import os; os.makedirs(FINAL_PROJECT, exist_ok=True)
print("runs ->", FINAL_PROJECT)

In [ ]:
import os
REPO_URL = "https://github.com/FooJames/DEEPLRN_Group2.git"
if not os.path.isdir("DEEPLRN_Group2"):
    !git clone $REPO_URL
else:
    !cd DEEPLRN_Group2 && git pull
%cd DEEPLRN_Group2

In [ ]:
from google.colab import userdata
import os
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
print("key loaded:", bool(os.environ.get("ROBOFLOW_API_KEY")))

In [ ]:
!python scripts/download_data.py --child-version 3 --hazard-version 1
!python scripts/fix_data_yaml.py data/child/data.yaml data/hazard/data.yaml
!python scripts/normalize_child_labels.py data/child

## Session A - child, ~0.9 h

Shorter, so it gets you a finished model quickly. Check the printed PLAN
line says *training from scratch* (or *resuming*), not *validating only*,
unless you intend a re-validation.

In [ ]:
!python scripts/train_final.py --model child --data data/child/data.yaml     --epochs 100 --project "$FINAL_PROJECT" 

In [ ]:
!zip -r final_child.zip results/metrics/final_child.csv "$FINAL_PROJECT"/final_child
!unzip -l final_child.zip | tail -4
from google.colab import files
files.download("final_child.zip")

## Session B - hazard, ~2.4 h

Also writes `results/metrics/per_class_hazard_final.csv` - the 12-class
table the proposal lists as a deliverable, regenerated under the pinned
ultralytics so it matches these exact weights.

In [ ]:
!python scripts/train_final.py --model hazard --data data/hazard/data.yaml     --epochs 100 --project "$FINAL_PROJECT" 

In [ ]:
!zip -r final_hazard.zip results/metrics/final_hazard.csv results/metrics/per_class_hazard_final.csv "$FINAL_PROJECT"/final_hazard
!unzip -l final_hazard.zip | tail -4
from google.colab import files
files.download("final_hazard.zip")

## Did the tuning help?

In [ ]:
import pandas as pd, os
for m in ("child", "hazard"):
    p = f"results/metrics/final_{m}.csv"
    if os.path.isfile(p):
        d = pd.read_csv(p).iloc[-1]
        print(f"{m:7} final {d.mAP50:.4f}/{d.mAP50_95:.4f}  "
              f"baseline {d.baseline_mAP50:.4f}/{d.baseline_mAP50_95:.4f}  "
              f"delta {d.delta_mAP50:+.4f}/{d.delta_mAP50_95:+.4f}")
print("
A negative delta is a legitimate result: it would mean the tuned")
print("config does not beat ultralytics' defaults at equal budget. Report it.")

### Notes
- Validation split only. Both test splits stay untouched until Phase 5.
- `infer_ms` is recorded per model; child@416 + hazard@640 together give the
  two-model-per-frame cost the proposal's introduction promises.
- If disconnected, re-run the same cell: a finished run is re-validated in
  minutes, an interrupted one continues from its checkpoint on Drive.